# Test-inference figure from a saved model

Given a **model checkpoint** and a **rendered surface-volume zarr** (e.g. `20260716083545` = the PHerc0813 test segment, or `20260115000000` = w044), this generates the per-depth **test figure** exactly like the training-time `add_test_figures` path — plus the **MAX-across-depths** collapsed panel.

It reuses the real pipeline (`create_model`, `predict_tiles`, the same normalization), so what you see matches training eval.

**Set the two paths in the CONFIG cell, then Run All.**

In [ ]:
# ============================ CONFIG — edit these ============================
MODEL_PATH = r"models/arch28/s07_v14_mil_deep_d8_p2.pth"   # checkpoint to run
SCROLL_ID  = 20260716083545                                # rendered zarr id (ves_zarrs2/<id>.zarr) + masks/<id>.png
ARCH       = "v14_mil_deep"                                # must match the checkpoint's architecture
TILE_SIZE  = 32                                            # tile size the model was trained at (32 or 24)
DEPTH      = 8                                             # depth window the model was trained at

SAVE_PNG   = None   # optional output path, e.g. r"C:/Users/ChenJeff/Documents/_ves_tmp/test_fig.png"; None = just show
# ============================================================================

In [ ]:
import os, sys
REPO = r"C:\Users\ChenJeff\Documents\vesuvius"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)  # so ./masks and ./ves_zarrs2 relative paths resolve

import numpy as np
import torch
import zarr
import matplotlib.pyplot as plt
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

from utils.config import Config
from utils.model import create_model
from utils.visualizer import predict_tiles, group_by_depth
from utils.dataloader import _load_unified_cache

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- build a minimal Config that matches the training data settings ---
c = Config()
c.data.tile_size = TILE_SIZE
c.data.depth = DEPTH
c.data.d_start = 0
c.data.d_end = 28
c.model.arch = ARCH
c.device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- open the rendered surface volume + its valid mask ---
vol = zarr.open(os.path.join(c.data.zarr_path, f'{SCROLL_ID}.zarr'), mode='r')
D, H, W = map(int, vol.shape)
mask = np.array(Image.open(f'./masks/{SCROLL_ID}.png').convert('L')) / 255.0
print(f'volume {SCROLL_ID}: shape=({D},{H},{W})  mask valid_frac={ (mask>0).mean():.3f }')

In [ ]:
# --- normalization stats: use the cached ones (same as training) ---
cache = _load_unified_cache()
st = cache.get(str(SCROLL_ID))
if st and all(k in st for k in ('mean', 'std', 'min', 'max')):
    g_mean, g_std, g_min, g_max = st['mean'], st['std'], st['min'], st['max']
    print(f'using cached norm: mean={g_mean:.3f} std={g_std:.3f} min={g_min:.3f} max={g_max:.3f}')
else:
    # fallback: compute z-score stats over valid voxels, then the post-zscore min/max
    print('no cached norm; computing from the volume (one-time)...')
    vals = []
    for d in range(D):
        sl = np.asarray(vol[d]).astype(np.float32); vals.append(sl[mask > 0])
    allv = np.concatenate(vals)
    g_mean, g_std = float(allv.mean()), float(allv.std())
    z = (allv - g_mean) / (g_std + 1e-8)
    g_min, g_max = float(z.min()), float(z.max())
    print(f'computed norm: mean={g_mean:.3f} std={g_std:.3f} min={g_min:.3f} max={g_max:.3f}')

In [ ]:
# --- load the model + weights (strict=False tolerates aux keys) ---
model, n_params = create_model(c)
sd = torch.load(MODEL_PATH, map_location=c.device)
if isinstance(sd, dict) and 'state_dict' in sd:
    sd = sd['state_dict']
missing, unexpected = model.load_state_dict(sd, strict=False)
model.eval()
print(f'loaded {MODEL_PATH}')
print(f'params={n_params:,}  missing={len(missing)} unexpected={len(unexpected)}')

In [ ]:
# --- generate valid tile coords over the whole fragment, grouped by depth window ---
def gen_tile_coords(D, H, W, mask, tile, depth):
    """mirror of TensorboardVisualizer._gen_tile_coords (half-depth z-step)."""
    z_span = max(0, D - depth + 1)
    y_span = max(0, H - tile + 1)
    x_span = max(0, W - tile + 1)
    z_step = max(1, depth // 2)
    coords = []
    for d in range(0, z_span, z_step):
        if d + depth > D:
            continue
        for y in range(0, y_span, tile):
            for x in range(0, x_span, tile):
                if np.sum(mask[y:y+tile, x:x+tile]) > 0:
                    coords.append((d, y, x))
    return coords

coords = gen_tile_coords(D, H, W, mask, TILE_SIZE, DEPTH)
grouped = group_by_depth(coords)
depths = sorted(grouped.keys())
print(f'{len(coords)} tiles across {len(depths)} depth windows: {depths}')

In [ ]:
# --- run prediction per depth window (sequential reads, same as training eval) ---
y_range = (0, H)
x_range = (0, W)
all_data = []
for d_start in depths:
    pred = predict_tiles(c, model, vol, mask, grouped[d_start], y_range, x_range,
                         d_start, f'Frag_{SCROLL_ID}', g_mean, g_std, g_min, g_max)
    all_data.append((pred, d_start, d_start + DEPTH))
    print(f'  depth {d_start}-{d_start+DEPTH}: pred {pred.shape}  '
          f'range [{np.nanmin(pred):.3f}, {np.nanmax(pred):.3f}]')

# MAX across all depth windows at each tile coordinate
import warnings
stack = np.stack([p for p, _, _ in all_data], axis=0)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    max_pred = np.nanmax(stack, axis=0)

In [ ]:
# --- render the test figure: one row per depth window + a final MAX row ---
n = len(all_data) + 1
hh, ww = all_data[0][0].shape
aspect = ww / max(hh, 1)
panel_w = max(6.0, min(14.0, ww * 0.08))
panel_h = max(1.5, min(10.0, panel_w / max(aspect, 1e-6)))

fig, axes = plt.subplots(n, 1, figsize=(panel_w, panel_h * n), squeeze=False)
for i, (pred, ds, de) in enumerate(all_data):
    ax = axes[i, 0]
    ax.imshow(pred, cmap='inferno', vmin=0, vmax=1, aspect='equal', interpolation='nearest')
    ax.set_title(f'Depth {ds}-{de}', fontsize=9); ax.axis('off')
ax = axes[n-1, 0]
ax.imshow(max_pred, cmap='inferno', vmin=0, vmax=1, aspect='equal', interpolation='nearest')
ax.set_title('MAX across all depths', fontsize=10, color='crimson'); ax.axis('off')
fig.suptitle(f'{ARCH}  |  frag {SCROLL_ID}  |  {os.path.basename(MODEL_PATH)}', fontsize=10)
plt.tight_layout()
if SAVE_PNG:
    fig.savefig(SAVE_PNG, dpi=130, bbox_inches='tight')
    print('saved ->', SAVE_PNG)
plt.show()

In [ ]:
# --- optional: standalone big view of just the MAX-across-depths map ---
plt.figure(figsize=(12, 12 / max(aspect, 1e-6)))
plt.imshow(max_pred, cmap='inferno', vmin=0, vmax=1, interpolation='nearest')
plt.title(f'MAX response — {SCROLL_ID}', fontsize=11); plt.axis('off')
plt.tight_layout(); plt.show()